# Cleaning the stock history
**SHUBHAM GUPTA**

I use the assignment table, `ohlc_hw`. The original tables are retained.
The main problems are repeated rows, HTML in ticker names, incomplete adjusted
prices and a set of unusually large adjustment factors. The cells below show
the exclusions before publishing `ohlc_hw_cleaned`.

This is retrospective data cleaning. The centered median used to flag
adjustment anomalies uses neighboring dates, so this table is intended for
descriptive reporting. It would need a causal rule for a trading backtest.

In [1]:
db_path = "data/data.db"

In [2]:
import html
import json
import re
import sqlite3
from html.parser import HTMLParser
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

path = Path(db_path)
if not path.is_file():
    raise FileNotFoundError(f"Place the course database at {path.resolve()}")
connection = sqlite3.connect(path)
raw = pd.read_sql_query("SELECT rowid AS source_rowid, * FROM ohlc_hw", connection)
display(raw.head())
display(raw.isna().sum().rename("missing").to_frame())
print(f"Raw rows: {len(raw):,}; raw ticker strings: {raw.ticker.nunique()}")

,source_rowid,ts,open,high,low,close,adj_close,volume,ticker
0,1,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,2,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,3,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,4,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,5,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


,missing
source_rowid,0
ts,0
open,0
high,0
low,0
close,0
adj_close,19855
volume,0
ticker,0


Raw rows: 397,711; raw ticker strings: 150


In [3]:
class TickerText(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts = []

    def handle_data(self, data):
        self.parts.append(data)


def normalize_ticker(value):
    if pd.isna(value):
        return ""
    parser = TickerText()
    parser.feed(html.unescape(str(value)))
    return "".join(parser.parts).strip().upper()


data = raw.copy()
mapping = {value: normalize_ticker(value) for value in raw.ticker.dropna().unique()}
data["ticker"] = data.ticker.map(mapping).fillna("")
data["ts"] = pd.to_datetime(data.ts, errors="coerce", format="mixed")
numeric = ["open", "high", "low", "close", "adj_close", "volume"]
data[numeric] = data[numeric].apply(pd.to_numeric, errors="coerce")
audit = [{"stage": "source rows", "removed": 0, "remaining": len(data)}]


def exclude(mask, reason):
    global data
    removed = int(mask.sum())
    data = data.loc[~mask].copy()
    audit.append({"stage": reason, "removed": removed, "remaining": len(data)})


exclude(data.ts.isna() | ~data.ticker.str.fullmatch(r"[A-Z0-9][A-Z0-9.\^-]*"),
        "invalid date or ticker")
exclude(data.ts.dt.dayofweek >= 5, "weekend observations")
exclude(~np.isfinite(data[numeric]).all(axis=1), "missing or nonfinite numbers")
prices = ["open", "high", "low", "close", "adj_close"]
exclude((data[prices] <= 0).any(axis=1) | (data.volume < 0)
        | (data.volume % 1 != 0), "invalid price or volume")
exclude((data.high < data[["open", "close", "low"]].max(axis=1))
        | (data.low > data[["open", "close", "high"]].min(axis=1)),
        "inconsistent OHLC bounds")
exclude(data.duplicated(["ticker", "ts", *numeric]), "identical duplicate quotes")

# Keep no arbitrary quote when values disagree at the same ticker/date.
exclude(data.duplicated(["ticker", "ts"], keep=False), "conflicting duplicate keys")
data = data.sort_values(["ticker", "ts"]).reset_index(drop=True)

factor = data.adj_close / data.close
local_factor = factor.groupby(data.ticker).transform(
    lambda series: series.rolling(21, center=True, min_periods=5).median()
)
relative_factor = factor / local_factor
suspicious = (relative_factor > 3) | (relative_factor < 1 / 3)
display(data.loc[suspicious, ["ticker", "ts", "close", "adj_close"]].head(10))
exclude(suspicious, "adjustment factor outside one-third to three times local median")
display(pd.DataFrame(audit))
print(f"Cleaned tickers: {data.ticker.nunique()}; latest date: {data.ts.max().date()}")

,ticker,ts,close,adj_close
33,AAPL,1981-02-03,0.123330,1.264587
89,AAPL,1981-04-28,0.126120,2.036653
97,AAPL,1981-05-08,0.125000,2.129290
149,AAPL,1981-07-29,0.106030,0.506329
155,AAPL,1981-08-06,0.112720,2.009828
159,AAPL,1981-08-13,0.103800,1.122224
160,AAPL,1981-08-14,0.102120,1.171347
197,AAPL,1981-10-08,0.082589,1.245215
250,AAPL,1981-12-31,0.098772,1.730921
259,AAPL,1982-01-15,0.089286,1.577125


,stage,removed,remaining
0,source rows,0,397711
1,invalid date or ticker,0,397711
2,weekend observations,413,397298
3,missing or nonfinite numbers,19836,377462
4,invalid price or volume,5,377457
5,inconsistent OHLC bounds,0,377457
6,identical duplicate quotes,27952,349505
7,conflicting duplicate keys,0,349505
8,adjustment factor outside one-third to three t...,10407,339098


Cleaned tickers: 30; latest date: 2024-12-06


The factor screen compares adjusted close divided by close with its
21-observation centered median. It quarantines extreme deviations beyond a
factor of three, without replacing them with unadjusted closes. It is a heuristic,
and valid corporate-action records could be excluded. Zero volume is allowed.
Missing prices are dropped rather than filled. Weekday market holidays are not
verified against an exchange calendar. Other plausible prices may still be wrong.

In [4]:
columns = ["ts", *numeric, "ticker"]
cleaned = data[columns].sort_values(["ticker", "ts"]).copy()
assert not cleaned.duplicated(["ticker", "ts"]).any()
assert cleaned[numeric].notna().all().all()
assert np.isfinite(cleaned[numeric]).all().all()
assert (cleaned[prices] > 0).all().all()
assert len(cleaned) > 0
cleaned["ts"] = cleaned.ts.dt.strftime("%Y-%m-%d")
cleaned["volume"] = cleaned.volume.astype("int64")

# Stage first, then replace the published table in one SQLite transaction.
cleaned.to_sql("ohlc_hw_cleaned_staging", connection, if_exists="replace", index=False)
try:
    connection.execute("BEGIN IMMEDIATE")
    connection.execute("DROP TABLE IF EXISTS ohlc_hw_cleaned")
    connection.execute("ALTER TABLE ohlc_hw_cleaned_staging RENAME TO ohlc_hw_cleaned")
    connection.execute(
        "CREATE UNIQUE INDEX cleaned_ticker_date ON ohlc_hw_cleaned(ticker, ts)"
    )
    connection.commit()
except Exception:
    connection.rollback()
    raise
count = connection.execute("SELECT COUNT(*) FROM ohlc_hw_cleaned").fetchone()[0]
assert count == len(cleaned)
connection.close()
display(cleaned.groupby("ticker").agg(observations=("ts", "size"), first=("ts", "min"),
                                      last=("ts", "max")))
Path("cleaning_audit.json").write_text(json.dumps(audit, indent=2) + "\n")
print(f"Published {count:,} quotes. Re-running this notebook refreshes the table.")

,observations,first,last
ticker,,,
AAPL,10146,1980-12-12,2024-10-18
AMGN,9631,1983-06-17,2024-10-18
AMZN,6409,1997-05-15,2024-12-06
AXP,12203,1972-04-03,2024-10-18
BA,14499,1962-01-02,2024-10-17
CAT,14495,1962-01-02,2024-10-18
CRM,4702,2004-06-23,2024-10-18
CSCO,8049,1990-02-16,2024-10-18
CVX,14562,1962-01-03,2024-12-06


Published 339,098 quotes. Re-running this notebook refreshes the table.
